# Costo Económico de los Trastornos Mentales (2020–2024)

**DOI:** [![DOI](https://img.shields.io/badge/DOI-10.7910%2FDVN%2FGOHAII-blue)](https://doi.org/10.7910/DVN/GOHAII)  
**Autor:** Juan Moises de la Serna | ORCID: [0000-0002-8401-8018](https://orcid.org/0000-0002-8401-8018)  
**Licencia:** CC0 1.0

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmoisesd/costo-economico-salud-mental-global-latam-2020-2024/blob/main/notebooks/costo_economico_salud_mental_analisis.ipynb)

## Descripción
Análisis del impacto económico de los trastornos mentales a nivel global y en América Latina, incluyendo pérdidas de productividad, costos sanitarios y carga en AVAD.

**Estadísticas clave (OMS/Banco Mundial 2020–2024):**
- Pérdida global en productividad: **1 billón USD/año**
- Costo estimado al 2030: **3,5% del PIB mundial**
- América Latina: 5–7% del PIB perdido por salud mental deficiente
- Depresión y ansiedad: **925.000 millones USD** de pérdida global (2020)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
print('✓ Bibliotecas cargadas correctamente')

In [ ]:
# ─── Conjunto de datos: costo por país/región ─────────────────────────────────
np.random.seed(42)
paises = {
    'Pais': ['EE.UU.','China','UE27','Japón','Brasil','México','Argentina','Colombia','Chile','Perú',
               'Reino Unido','Alemania','Francia','India','Australia','Canadá','Corea del Sur','Nigeria','Sudáfrica','Arabia Saudita'],
    'Region': ['América del Norte','Asia Oriental','Europa','Asia Oriental','América Latina','América Latina','América Latina',
                'América Latina','América Latina','América Latina','Europa','Europa','Europa','Asia Meridional',
                'Oceanía','América del Norte','Asia Oriental','África Subsahariana','África Subsahariana','Oriente Medio'],
    'PIB_miles_millones_USD': [23000,17700,17400,4900,1870,1290,630,340,310,230,2960,4260,2940,3180,1700,2000,1800,510,420,900],
    'costo_SM_pct_PIB': [3.2,2.8,2.9,2.5,4.8,5.1,5.5,5.3,4.2,5.8,3.1,2.7,2.8,3.5,2.9,3.0,2.6,6.2,5.9,2.4]
}
df_paises = pd.DataFrame(paises)
df_paises['costo_SM_miles_millones'] = df_paises['PIB_miles_millones_USD'] * df_paises['costo_SM_pct_PIB'] / 100

# Serie temporal 2020-2024
anios = list(range(2020, 2025))
costo_global = [830, 860, 900, 945, 980]  # miles de millones USD
costo_latam  = [118, 127, 138, 149, 161]  # miles de millones USD
perdida_productividad = [625, 651, 680, 715, 745]

df_tendencia = pd.DataFrame({'anio': anios, 'costo_global_mmm': costo_global,
                               'costo_latam_mmm': costo_latam, 'perdida_productividad_mmm': perdida_productividad})

print('Conjunto de datos de países:', df_paises.shape)
print(df_paises[['Pais','Region','costo_SM_pct_PIB','costo_SM_miles_millones']].round(1).to_string(index=False))

In [ ]:
# ─── Figura 1: Costo % PIB por País ───────────────────────────────────────────
fig, eje = plt.subplots(figsize=(14, 7))
colores = ['#E74C3C' if r=='América Latina' else '#3498DB' if r=='Europa' else
            '#2ECC71' if r=='América del Norte' else '#F39C12' for r in df_paises['Region']]
df_ord = df_paises.sort_values('costo_SM_pct_PIB', ascending=True)
bars = eje.barh(df_ord['Pais'], df_ord['costo_SM_pct_PIB'],
                color=[colores[i] for i in df_ord.index], edgecolor='white', linewidth=0.5)
media = df_paises['costo_SM_pct_PIB'].mean()
eje.axvline(x=media, color='black', linestyle='--', lw=1.5, label=f'Media global: {media:.1f}%')
parches = [mpatches.Patch(color='#E74C3C', label='América Latina'),
            mpatches.Patch(color='#3498DB', label='Europa'),
            mpatches.Patch(color='#2ECC71', label='América del Norte'),
            mpatches.Patch(color='#F39C12', label='Otras regiones')]
eje.legend(handles=parches, loc='lower right', fontsize=9)
eje.set_xlabel('Costo Económico de la Salud Mental (% del PIB)', fontsize=12)
eje.set_title('Costo Económico de los Trastornos Mentales por País (% del PIB)\nDOI: 10.7910/DVN/GOHAII', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_costo_pct_pib.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Figura 2: Tendencia Global vs LATAM 2020–2024 ───────────────────────────
fig, eje1 = plt.subplots(figsize=(10, 6))
eje2 = eje1.twinx()

eje1.plot(df_tendencia['anio'], df_tendencia['costo_global_mmm'], 'b-o', lw=2.5, markersize=8,
           label='Costo global (miles de millones USD)', color='#2980B9')
eje1.fill_between(df_tendencia['anio'], df_tendencia['costo_global_mmm'], alpha=0.12, color='#2980B9')
eje2.plot(df_tendencia['anio'], df_tendencia['costo_latam_mmm'], 'r-s', lw=2.5, markersize=8,
           label='América Latina (miles de millones USD)', color='#E74C3C')
eje2.fill_between(df_tendencia['anio'], df_tendencia['costo_latam_mmm'], alpha=0.12, color='#E74C3C')

eje1.set_xlabel('Año', fontsize=12)
eje1.set_ylabel('Costo Global (miles de millones USD)', color='#2980B9', fontsize=11)
eje2.set_ylabel('Costo América Latina (miles de millones USD)', color='#E74C3C', fontsize=11)
eje1.set_title('Tendencia del Costo Económico en Salud Mental: 2020–2024\nDOI: 10.7910/DVN/GOHAII', fontsize=13, fontweight='bold')

líneas1, etiq1 = eje1.get_legend_handles_labels()
líneas2, etiq2 = eje2.get_legend_handles_labels()
eje1.legend(líneas1+líneas2, etiq1+etiq2, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig('fig2_tendencia_global_latam.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Estadísticas clave ───────────────────────────────────────────────────────
print('=== ESTADÍSTICAS GLOBALES ===')
print(f'Total de países analizados: {len(df_paises)}')
print(f'Costo global en SM (2024): ${costo_global[-1]:,} miles de millones USD')
print(f'Costo LATAM en SM (2024):  ${costo_latam[-1]:,} miles de millones USD')
print(f'% PIB promedio (todos):    {df_paises["costo_SM_pct_PIB"].mean():.2f}%')
print(f'% PIB promedio (LATAM):    {df_paises[df_paises["Region"]=="América Latina"]["costo_SM_pct_PIB"].mean():.2f}%')
print(f'\nCrecimiento 2020-2024 (global): {((costo_global[-1]/costo_global[0])-1)*100:.1f}%')
print(f'Crecimiento 2020-2024 (LATAM):  {((costo_latam[-1]/costo_latam[0])-1)*100:.1f}%')

## Cita
```bibtex
@data{DVN/GOHAII_2024,
  author    = {de la Serna, Juan Moises},
  title     = {Costo Económico de los Trastornos Mentales: Datos Globales y América Latina 2020-2024},
  year      = {2024},
  publisher = {Harvard Dataverse},
  doi       = {10.7910/DVN/GOHAII},
  url       = {https://doi.org/10.7910/DVN/GOHAII}
}
```
## Licencia: CC0 1.0 Universal — Dominio Público